<a href="https://colab.research.google.com/github/yesahek/CatsVsDogs-Image-Classifier/blob/main/notebooks/data_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import random
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score,
    confusion_matrix, classification_report,
)

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow version : 2.20.0
Keras version      : 3.13.2
GPU available      : True


In [29]:
print("🔗 Connecting to your Google Drive...")
drive.mount('/content/drive')

# Create a permanent dedicated folder for your project inside your Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/cats_vs_dogs_classifier"
RAW_DATA_DIR = f"{DRIVE_PROJECT_PATH}/data/raw"
os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
os.makedirs(f"{DRIVE_PROJECT_PATH}/models", exist_ok=True)
os.makedirs(f"{DRIVE_PROJECT_PATH}/docs", exist_ok=True)

print(f"\n✅ Google Drive connected! Permanent project folder ready at:\n📂 {DRIVE_PROJECT_PATH}")

🔗 Connecting to your Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Google Drive connected! Permanent project folder ready at:
📂 /content/drive/MyDrive/cats_vs_dogs_classifier


# Connecting Kaggle

In [38]:

from google.colab import files

print("📁 Select your kaggle.json file from your computer...")
uploaded = files.upload()

# Move it to the right location so the kaggle CLI can find it
os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("/content/kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)   # secure the file

print("\n✅ kaggle.json uploaded and configured!")

📁 Select your kaggle.json file from your computer...


Saving kaggle.json to kaggle (6).json

✅ kaggle.json uploaded and configured!


In [50]:
## Temporaty storage
# !kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data --unzip

# # Check what was downloaded
# print("\n📂 Downloaded files:")
# for p in sorted(Path("/content/data").rglob("*"))[:20]:
#     indent = "  " * (len(p.parts) - 3)
#     print(f"{indent}{'📁' if p.is_dir() else '🖼️ '} {p.name}")

In [39]:
!pip install -q kaggle
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p {RAW_DATA_DIR} --unzip

# Check what was downloaded
print("\n📂 Downloaded files:")
for p in sorted(Path(RAW_DATA_DIR).rglob("*"))[:20]:
    indent = "  " * (len(p.parts) - 3)
    print(f"{indent}{'📁' if p.is_dir() else '🖼️ '} {p.name}")

Dataset URL: https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset
License(s): other
100% 788M/788M [00:09<00:00, 90.0MB/s]


📂 Downloaded files:
          🖼️  MSR-LA - 3467.docx
          📁 PetImages
            📁 Cat
              🖼️  0.jpg
              🖼️  1.jpg
              🖼️  10.jpg
              🖼️  100.jpg
              🖼️  1000.jpg
              🖼️  10000.jpg
              🖼️  10001.jpg
              🖼️  10002.jpg
              🖼️  10003.jpg
              🖼️  10004.jpg
              🖼️  10005.jpg
              🖼️  10006.jpg
              🖼️  10007.jpg
              🖼️  10008.jpg
              🖼️  10009.jpg
              🖼️  1001.jpg
              🖼️  10010.jpg


# Finding the Dataset

In [41]:
#finds where Cat/ and Dog/ acctually are
# Detect the folder structure
import glob
# for temp storage
#data_root = Path('/content/data')
data_root = Path(RAW_DATA_DIR)

# Search for Cat and Dog folders (case-insensitive)
cat_dirs = list(data_root.rglob("[Cc]at")) + list(data_root.rglob("[Cc]ats"))
dog_dirs = list(data_root.rglob("[Dd]og")) + list(data_root.rglob("[Dd]ogs"))

In [42]:
cat_dirs = [d for d in cat_dirs if d.is_dir()]
dog_dirs = [d for d in dog_dirs if d.is_dir()]

In [43]:
print("Found Cat folders: ", cat_dirs)
print("Found Dog folders: ", dog_dirs)

Found Cat folders:  [PosixPath('/content/drive/MyDrive/cats_vs_dogs_classifier/data/raw/PetImages/Cat')]
Found Dog folders:  [PosixPath('/content/drive/MyDrive/cats_vs_dogs_classifier/data/raw/PetImages/Dog')]


In [44]:
RAW_CAT_DIR = cat_dirs[0]
RAW_DOG_DIR = dog_dirs[0]
RAW_DIR = RAW_CAT_DIR.parent
print(f'\nRaw Cat folder {RAW_CAT_DIR}')
print(f'\nRaw Dog folder {RAW_DOG_DIR}')
print(f"\nRaw root       {RAW_DIR}")
print(f"\n     Cat images: {len(list(RAW_CAT_DIR.iterdir()))}")
print(f"\n     Dog images: {len(list(RAW_DOG_DIR.iterdir()))}")


Raw Cat folder /content/drive/MyDrive/cats_vs_dogs_classifier/data/raw/PetImages/Cat

Raw Dog folder /content/drive/MyDrive/cats_vs_dogs_classifier/data/raw/PetImages/Dog

Raw root       /content/drive/MyDrive/cats_vs_dogs_classifier/data/raw/PetImages

     Cat images: 12501

     Dog images: 12501
